## Daily Challenge: Fine-Tuning LLMs with LoRA

PEFT (Parameter-Efficient Fine-Tuning) methods like **LoRA** allow us to fine-tune large language models by updating only a small subset of parameters, drastically reducing compute and memory costs.

In this challenge we will:
1. Install required libraries
2. Load `bigscience/bloomz-560m` and its tokenizer
3. Load and preprocess 10% of `Abiate/english_quotes`
4. Configure and apply LoRA with `LoraConfig` + `get_peft_model`
5. Train with `TrainingArguments` + `Trainer`
6. Save the fine-tuned LoRA model
7. Load it back with `PeftModel.from_pretrained` and generate text


## Step 1 — Install required libraries

In [ ]:
import os

# Install compatible versions of all required libraries
# peft==0.4.0 is specified in the challenge instructions
%pip install --quiet peft==0.4.0 datasets transformers accelerate

# Create cache directory for model outputs
os.makedirs("cache", exist_ok=True)
print("Libraries installed and cache directory created.")


## Step 2 — Load pre-trained model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

model_name = "bigscience/bloomz-560m"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Load foundation model (causal LM for text generation)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)

# Define output directory for saving the fine-tuned LoRA model
output_directory = os.path.join("cache", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)

print(f"Model '{model_name}' loaded successfully.")
print(f"Output directory: {output_directory}")


## Step 3 — Load and preprocess the dataset

We load the `Abiate/english_quotes` dataset and **sample 10%** of the training split, then tokenize the `quote` column for causal language modelling.


In [ ]:
from datasets import load_dataset

# Load 10% of the training split of Abiate/english_quotes
# split="train[:10%]" is the idiomatic way to sample 10% with the datasets library
data = load_dataset("Abiate/english_quotes", split="train[:10%]")

print(f"Dataset loaded: {len(data)} examples (10% of training split)")
print(f"Columns: {data.column_names}")
print(f"\nSample entry:\n{data[0]}")

# Set pad token to eos token (BLOOM has no dedicated pad token)
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the 'quote' column — this is the target field specified in the challenge
data = data.map(
    lambda samples: tokenizer(samples["quote"], truncation=True, padding="max_length", max_length=128),
    batched=True
)

# Keep only the first 5 examples for quick training (train_sample)
train_sample = data.select(range(5))
display(train_sample.to_pandas())


## Step 4 — Configure LoRA

`LoraConfig` controls the low-rank decomposition:
- `r=1` — rank of the low-rank matrices (lower = fewer trainable params)
- `lora_alpha=1` — scaling factor for the LoRA update
- `target_modules` — which sub-modules to apply LoRA to (`query_key_value` for BLOOM)
- `lora_dropout=0.05` — regularisation on LoRA paths
- `task_type="CAUSAL_LM"` — needed so PEFT knows the output head structure


In [ ]:
import peft
from peft import LoraConfig, get_peft_model

# r=1 as specified in the challenge hint
lora_config = LoraConfig(
    r=1,                             # rank of the low-rank matrices
    lora_alpha=1,                    # scaling factor (typically set equal to r)
    target_modules=["query_key_value"],  # BLOOM attention projection
    lora_dropout=0.05,               # dropout probability on LoRA paths
    bias="none",                     # do not train bias parameters
    task_type="CAUSAL_LM"            # causal language modelling
)

# Wrap the foundation model with LoRA adapter layers
peft_model = get_peft_model(foundation_model, lora_config)

# Confirm that only LoRA parameters are trainable
print(peft_model.print_trainable_parameters())


## Step 5 — Set up training arguments and train

`TrainingArguments` configures the training loop. `Trainer` handles the full training pipeline including gradient updates and logging.


In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

training_args = TrainingArguments(
    report_to="none",                # disable external logging (wandb, etc.)
    output_dir=output_directory,
    auto_find_batch_size=True,       # automatically finds a suitable batch size
    learning_rate=3e-2,              # higher LR than full fine-tuning (typical for LoRA)
    num_train_epochs=1,              # 1 epoch is enough for this demo
    use_cpu=True                     # force CPU (no GPU required)
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,      # 5-example sample for quick training
    data_collator=transformers.DataCollatorForLanguageModeling(
        tokenizer, mlm=False         # mlm=False → causal LM (next-token prediction)
    )
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("Training complete.")


## Step 6 — Save the fine-tuned LoRA model

In [ ]:
import time

# Timestamp the saved model path to avoid collisions between runs
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Save only the lightweight LoRA adapter weights (not the full model)
trainer.model.save_pretrained(peft_model_path)

print(f"LoRA adapter saved to: {peft_model_path}")
print(f"Files saved: {os.listdir(peft_model_path)}")


## Step 7 — Load the saved LoRA model and generate text

`PeftModel.from_pretrained` reattaches the saved LoRA adapter to the frozen foundation model. Setting `is_trainable=False` ensures the adapter is locked for pure inference.


In [ ]:
from peft import PeftModel

# Load the PEFT model: foundation model + saved LoRA adapter
# is_trainable=False → inference mode, no further training
peft_model_loaded = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)

# Move model to CPU (consistent with training config)
peft_model_loaded = peft_model_loaded.to('cpu')
peft_model_loaded.eval()

# Generate output tokens
prompt = "Two things are infinite: "
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to('cpu') for k, v in inputs.items()}

outputs = peft_model_loaded.generate(
    input_ids=inputs["input_ids"],
    max_new_tokens=50,        # generate up to 50 new tokens
    do_sample=True,           # sampling for diverse outputs
    top_k=50,                 # consider only top-50 tokens at each step
    top_p=0.95,               # nucleus sampling threshold
    temperature=0.7,          # lower temperature = more focused output
    eos_token_id=tokenizer.eos_token_id
)

print("Prompt:", prompt)
print("Generated text:")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))
